# Broadcast → .srt transcription (Kaggle T4 GPU)

Runs the `broadcast-srt` pipeline (faster-whisper `large-v3-turbo`, per-chunk language
detection) on a Darija / Arabic / French broadcast clip and verifies the output `.srt`.

**Setup:** Settings → Accelerator → **GPU T4 x2** (or P100). Internet **On** (first run
downloads the model weights). The repo is cloned automatically — just run all cells.
To use your own `.mp3`, upload it via **Add Data** → Upload and set `CLIP` manually.

## 1. Install

In [ ]:
!pip -q install "faster-whisper>=1.1.0" "transformers>=4.46.0" "peft>=0.14.0"

## 2. Clone the repo & load pipeline code

The notebook auto-clones the transcription repo so you get the latest pipeline code
and a sample broadcast clip. No manual dataset upload needed.

In [ ]:
import os, sys
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/haca-transcription")
if not REPO_ROOT.exists():
    !git clone --depth 1 https://github.com/MarTCM/haca-transcription.git "{REPO_ROOT}"
else:
    print("[repo] already cloned")

sys.path.insert(0, str(REPO_ROOT / "src"))
import transcribe
from srt_writer import write_srt
print("[repo]", REPO_ROOT)

## 3. Choose a clip

By default, uses the trimmed broadcast clip from the repo. To transcribe your own
audio, upload it via **Add Data** → Upload, then set `CLIP` to its path below.

In [ ]:
import glob

# --- set this to your uploaded file, or leave the default ---
CLIP = str(REPO_ROOT / "samples" / "mobachara_ma3akom_trimmed.mp3")

# Also list user-uploaded media under /kaggle/input for convenience.
uploaded = sorted(
    p for p in glob.glob("/kaggle/input/**/*", recursive=True)
    if os.path.splitext(p)[1].lower() in transcribe.MEDIA_EXTS
)
if uploaded:
    print("uploaded media files:")
    for p in uploaded:
        print("  ", p)

assert os.path.exists(CLIP), f"clip not found: {CLIP}"
print("\nusing CLIP:", CLIP)

## 4. Load the model on GPU

`large-v3-turbo` in `float16`. First run downloads the weights (~3 GB).
For best Darija quality, flip the toggle below to enable the anaszil LoRA adapter
(see also `kaggle_compare_darija_models2.ipynb`).

In [ ]:
# --- toggle: set to True to route Arabic chunks through the anaszil LoRA ---
USE_DARIJA_LORA = False

model = transcribe.load_model("large-v3-turbo", device="cuda", compute_type="float16")
lora_pipe = None
if USE_DARIJA_LORA:
    lora_pipe = transcribe._load_darija_lora(device="cuda")

## 5. Transcribe with per-chunk language detection

`lang="auto"` detects the language per ~25 s chunk (allow-list `ar,fr,en`), so a
code-switched broadcast transcribes natively.

In [ ]:
segments = transcribe.transcribe_file(
    CLIP, model,
    lang="auto", allowed=("ar", "fr", "en"),
    max_chunk_s=25.0, beam_size=5,
    darija_lora=USE_DARIJA_LORA, lora_pipe=lora_pipe,
)
print(f"{len(segments)} cues")
for s in segments[:12]:
    print(f"[{s['start']:6.1f}-{s['end']:6.1f}] ({s['lang']}) {s['text']}")

# Per-language cue counts — sanity check that a mixed clip actually switched.
from collections import Counter
print("lang mix:", Counter(s["lang"] for s in segments))

## 6. Write the .srt

In [ ]:
out_path = write_srt(segments, f"/kaggle/working/{os.path.splitext(os.path.basename(CLIP))[0]}.srt")
print("wrote", out_path)
print(out_path.read_text(encoding="utf-8")[:1500])

## 7. Verify the output is a valid, parseable SRT

Re-parse with the same standard-SRT contract a downstream consumer uses (e.g. the HACA
benchmark's `srt_utils.parse_srt`). If the cue count matches, the file is interoperable.
Download it from the Kaggle **Output** panel (`/kaggle/working/...srt`).

In [ ]:
import re
BLOCK = re.compile(r"(\d+)\n(\d\d:\d\d:\d\d,\d\d\d) --> (\d\d:\d\d:\d\d,\d\d\d)\n(.+?)(?=\n\n|\Z)", re.DOTALL)
text = out_path.read_text(encoding="utf-8")
cues = BLOCK.findall(text)
print(f"parsed {len(cues)} cues; first:", cues[0] if cues else None)
assert len(cues) == len(segments), "cue count mismatch — SRT format problem"